In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

# Load the cleaned dataframe exported from credit_risk_investigation.ipynb
df = pd.read_csv("data/credit_risk_dataset_cleaned.csv")
df.head()


# Logistic Regression Baseline

This notebook trains and evaluates an interpretable logistic-regression baseline for binary loan-default prediction. The cleaned dataset is produced by `credit_risk_investigation.ipynb`.

`X` contains applicant and loan information; `y` is `loan_status`, where 1 indicates default and 0 indicates repayment.

In [ ]:
X = df.drop("loan_status", axis=1)
y = df["loan_status"]

print("Features:", X.shape)
print("Target:", y.shape)
print(f"Default rate: {y.mean():.1%}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Training default rate: {y_train.mean():.1%}")
print(f"Test default rate: {y_test.mean():.1%}")

The dataset has an imbalanced target, so `stratify=y` preserves the default rate across the train/test split.

## Preprocessing

In [ ]:
categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_features = X.select_dtypes(
    exclude=["object"]
).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numerical_features
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

# Logistic Regression 

This will be the baseline model. Logistic regression was used as it is interpretable and relatively simple, appropriate for binary classification.
Note the `class_weight="balanced"` used to give greater weight to the minority class as there was an imbalance.

In [ ]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

In [ ]:
# Train the model
logistic_model.fit(X_train, y_train)
logistic_predictions = logistic_model.predict(X_test)
logistic_probabilities = logistic_model.predict_proba(X_test)[:, 1]

`predict` returns binary class labels, while `predict_proba` returns the estimated probability of default.

For credit risk, probabilities are useful because a lender may want a risk estimate rather than only a default/no-default decision.

## Evaluation

In [ ]:
metrics = pd.DataFrame(
    {
        "metric": ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"],
        "value": [
            accuracy_score(y_test, logistic_predictions),
            precision_score(y_test, logistic_predictions, zero_division=0),
            recall_score(y_test, logistic_predictions, zero_division=0),
            f1_score(y_test, logistic_predictions, zero_division=0),
            roc_auc_score(y_test, logistic_probabilities),
        ],
    }
).set_index("metric")

print(metrics.round(3))

cm = confusion_matrix(y_test, logistic_predictions)
cm_df = pd.DataFrame(
    cm,
    index=["Actual: No Default", "Actual: Default"],
    columns=["Predicted: No Default", "Predicted: Default"]
)
print("\nConfusion matrix:")
print(cm_df)

In [ ]:
print(classification_report(y_test, logistic_predictions))

Accuracy provides an overall measure of classification performance, but recall is particularly relevant to credit-risk modelling because failing to identify a borrower who subsequently defaults may result in financial losses.

Recall is not always the most important metric. Instead, metric selection depends on business costs.


# Random Forest Model

Random Forest is an ensemble of decision trees, and unlike logistic regression it can capture nonlinear relationships and interactions between features without them being specified manually. It's evaluated here as a stronger, less interpretable alternative to the logistic regression baseline.


In [ ]:
random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)


In [ ]:
random_forest_model.fit(X_train, y_train)

rf_predictions = random_forest_model.predict(X_test)

rf_probabilities = random_forest_model.predict_proba(X_test)[:, 1]

In [ ]:
print("Random Forest")

print("Accuracy:",
      round(accuracy_score(y_test, rf_predictions), 3))

print("Precision:",
      round(precision_score(y_test, rf_predictions), 3))

print("Recall:",
      round(recall_score(y_test, rf_predictions), 3))

print("F1:",
      round(f1_score(y_test, rf_predictions), 3))

print("ROC-AUC:",
      round(roc_auc_score(y_test, rf_probabilities), 3))

In [ ]:
print(classification_report(y_test, rf_predictions))

## Model Comparison

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, logistic_predictions),
        accuracy_score(y_test, rf_predictions)
    ],
    "Precision": [
        precision_score(y_test, logistic_predictions),
        precision_score(y_test, rf_predictions)
    ],
    "Recall": [
        recall_score(y_test, logistic_predictions),
        recall_score(y_test, rf_predictions)
    ],
    "F1": [
        f1_score(y_test, logistic_predictions),
        f1_score(y_test, rf_predictions)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, logistic_probabilities),
        roc_auc_score(y_test, rf_probabilities)
    ]
})

results.round(3)

### Results
Overall, Random forest performaned better, with it doing better in 4 out of the 5 metrics. Although Logistic Regression achieved slightly higher recall, the Random Forest's largely higher precision indicates that its positive predictions were more reliable.

But model selection in a real credit-risk setting would depend on the relative costs of false positives and false negatives. If the main objective were to minimise missed defaults, the higher recall of Logistic Regression would be preferable, potentially after adjusting the classification threshold.

## Confusion Matrix

The confusion matrix below shows how the Random Forest's predictions break down against the actual outcomes:

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    rf_predictions
)

plt.title("Random Forest Confusion Matrix")
plt.show()


The Random Forest correctly classified the majority of both non-defaulting and defaulting borrowers. It correctly identified 4,838 non-defaults and 1,032 defaults. However, it failed to identify 356 borrowers who subsequently defaulted, compared with 109 non-defaulting borrowers incorrectly classified as defaults. This indicates that the model is relatively conservative when identifying defaults: its predictions of default are highly reliable, but it still misses a meaningful proportion of actual defaults.

### ROC curves

The ROC curve considers many possible classification thresholds, showing how each model's true-positive rate trades off against its false-positive rate.

In [ ]:
RocCurveDisplay.from_predictions(
    y_test,
    logistic_probabilities,
    name="Logistic Regression"
)

RocCurveDisplay.from_predictions(
    y_test,
    rf_probabilities,
    name="Random Forest"
)

plt.title("ROC Curve Comparison")
plt.show()


The ROC curves indicate that both models demonstrate strong discriminatory performance, with Logistic Regression achieving an AUC of 0.87 and Random Forest achieving an AUC of 0.93. The Random Forest curve remains closer to the top-left corner across most thresholds, indicating that it generally achieves a higher true-positive rate for a given false-positive rate. This suggests that the Random Forest is better able to distinguish between defaulting and non-defaulting borrowers.

This finding is consistent with the model comparison, where Random Forest achieved higher accuracy, precision, F1-score and ROC-AUC. However, Logistic Regression achieved slightly higher recall (0.774 compared with 0.744), meaning it identified a marginally greater proportion of actual defaults at the selected classification threshold.

## Feature Importance

Feature importance analysis indicates that loan-to-income ratio, borrower income, interest rate and loan amount are the most influential features in the Random Forest model. This suggests that financial characteristics of the borrower and loan provide the majority of the predictive information used by the model. However, feature importance represents the contribution of variables to the model's predictions rather than causal relationships. Furthermore, correlated variables may distribute importance between one another. The relatively high importance of loan interest rate also warrants further investigation, as this variable may contain information related to prior credit-risk assessment.

In [ ]:
feature_names = (
    random_forest_model
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

In [ ]:
importances = (
    random_forest_model
    .named_steps["classifier"]
    .feature_importances_
)

In [ ]:
feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(
    "importance",
    ascending=False
)

In [ ]:
feature_importance.head(15)

In [ ]:
feature_importance.head(15).sort_values(
    "importance"
).plot(
    x="feature",
    y="importance",
    kind="barh",
    figsize=(10, 6),
    legend=False
)

plt.title("Top 15 Features - Random Forest")
plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.show()

Feature importance shows what information the model is actually using to make its predictions.

The analysis indicates that loan-to-income ratio, borrower income, interest rate and loan amount are the most influential features in the Random Forest model. This suggests that the financial characteristics of the borrower and the loan provide the majority of the predictive information used by the model. However, feature importance represents the contribution of variables to the model's predictions rather than a causal relationship, and correlated variables may split importance between one another. The relatively high importance of loan interest rate also warrants further investigation, as this variable may already encode information from a prior credit-risk assessment.

# Conclusion

Both models are able to distinguish defaulting from non-defaulting borrowers well above chance, but they make different trade-offs. Random Forest is the stronger model overall (higher accuracy, precision, F1 and ROC-AUC), and its feature importances point to loan-to-income ratio, income, interest rate and loan amount as the main drivers of predicted risk. Logistic Regression trails on most metrics but offers a directly interpretable, coefficient-based view of risk and achieves marginally higher recall, the metric most directly tied to catching defaults before they happen.

In practice, the right model — and the right classification threshold — would be chosen based on the relative cost of a missed default versus a rejected good borrower, rather than on accuracy alone. Natural next steps would include tuning the classification threshold against a specific cost function, cross-validating both models rather than relying on a single train/test split, and investigating whether `loan_int_rate` is leaking information from an earlier credit decision.
